In [0]:
spark.table("ml.data.silver_turbine_metadata").write.format("json").mode("overwrite").save("/Volumes/ml/data/dataset/test/json_silver_turbine_metadata")
spark.table("ml.data.silver_scada_hourly").write.format("json").mode("overwrite").save("/Volumes/ml/data/dataset/test/json_silver_scada_hourly")
spark.table("ml.data.silver_failure_logs").write.format("json").mode("overwrite").save("/Volumes/ml/data/dataset/test/json_silver_failure_logs")

#  This cell installs LightGBM and XGBoost libraries in your Databricks cluster. -->


In [0]:
# Install lightgbm in your cluster
%pip install lightgbm xgboost

# The following cell loads an MLflow model, selects features from the  turbine dataset, generates predictions, and joins them back to the Spark DataFrame for display.

In [0]:


import mlflow.pyfunc
def load_model(model_uri):
    return mlflow.pyfunc.load_model(model_uri)

def get_features():
    return [
        'power_kw_avg_mean_6h', 'power_kw_avg_mean_12h', 'power_kw_avg_mean_24h', 'power_kw_avg_mean_72h',
        'wind_speed_m_s_avg_mean_6h', 'wind_speed_m_s_avg_mean_12h', 'wind_speed_m_s_avg_mean_24h', 'wind_speed_m_s_avg_mean_72h',
        'vibration_mm_s_avg_mean_6h', 'vibration_mm_s_avg_mean_12h', 'vibration_mm_s_avg_mean_24h', 'vibration_mm_s_avg_mean_72h',
        'gearbox_temp_c_avg_mean_6h', 'gearbox_temp_c_avg_mean_12h', 'gearbox_temp_c_avg_mean_24h', 'gearbox_temp_c_avg_mean_72h',
        'generator_speed_rpm_avg_mean_6h', 'generator_speed_rpm_avg_mean_12h', 'generator_speed_rpm_avg_mean_24h', 'generator_speed_rpm_avg_mean_72h',
        'generator_temp_c_avg_mean_6h', 'generator_temp_c_avg_mean_12h', 'generator_temp_c_avg_mean_24h', 'generator_temp_c_avg_mean_72h',
        'nacelle_temp_c_avg_mean_6h', 'nacelle_temp_c_avg_mean_12h', 'nacelle_temp_c_avg_mean_24h', 'nacelle_temp_c_avg_mean_72h',
        'power_kw_avg_stddev_6h', 'power_kw_avg_stddev_12h', 'power_kw_avg_stddev_24h', 'power_kw_avg_stddev_72h',
        'wind_speed_m_s_avg_stddev_6h', 'wind_speed_m_s_avg_stddev_12h', 'wind_speed_m_s_avg_stddev_24h', 'wind_speed_m_s_avg_stddev_72h',
        'vibration_mm_s_avg_stddev_6h', 'vibration_mm_s_avg_stddev_12h', 'vibration_mm_s_avg_stddev_24h', 'vibration_mm_s_avg_stddev_72h',
        'gearbox_temp_c_avg_stddev_6h', 'gearbox_temp_c_avg_stddev_12h', 'gearbox_temp_c_avg_stddev_24h', 'gearbox_temp_c_avg_stddev_72h',
        'generator_speed_rpm_avg_stddev_6h', 'generator_speed_rpm_avg_stddev_12h', 'generator_speed_rpm_avg_stddev_24h', 'generator_speed_rpm_avg_stddev_72h',
        'generator_temp_c_avg_stddev_6h', 'generator_temp_c_avg_stddev_12h', 'generator_temp_c_avg_stddev_24h', 'generator_temp_c_avg_stddev_72h',
        'nacelle_temp_c_avg_stddev_6h', 'nacelle_temp_c_avg_stddev_12h', 'nacelle_temp_c_avg_stddev_24h', 'nacelle_temp_c_avg_stddev_72h'
    ]

def predict_and_join(model, dfs, features_cols):
    pdf = dfs.select(features_cols).toPandas()
    predictions = model.predict(pdf)
    from pyspark.sql import functions as F
    pred_df = spark.createDataFrame(
        pdf.assign(prediction=predictions)
    )
    final_df = dfs.withColumn("row_id", F.monotonically_increasing_id()) \
        .join(
            pred_df.withColumn("row_id", F.monotonically_increasing_id()),
            on="row_id",
            how="inner"
        ).drop("row_id")
    return final_df

model_uri = "models:/ml.models.wind_turbine_failure_turned_24h/1"
model = load_model(model_uri)
dfs = spark.read.table("ml.data.model_24h")
features_cols = get_features()
final_df = predict_and_join(model, dfs, features_cols)
display(final_df)

In [0]:
model_uri = "models:/ml.models.wind_turbine_failure_24h/1"
model = load_model(model_uri)
dfs = spark.read.table("ml.data.model_24h")
features_cols = get_features()
final_df = predict_and_join(model, dfs, features_cols)
display(final_df)

In [0]:
model_uri = "models:/ml.models.wind_turbine_failure_72h/1"
model = load_model(model_uri)
dfs = spark.read.table("ml.data.model_72h")
features_cols = get_features()
final_df = predict_and_join(model, dfs, features_cols)
display(final_df)

In [0]:
model_uri = "models:/ml.models.wind_turbine_failure_turned_72h/1"
model = load_model(model_uri)
dfs = spark.read.table("ml.data.model_72h")
features_cols = get_features()
final_df = predict_and_join(model, dfs, features_cols)
display(final_df)

In [0]:
display(final_df)